# ÔN TẬP: CÁC THUẬT TOÁN KHAI THÁC DỮ LIỆU

**Bài 1 — Các dạng thuật toán:** luật kết hợp (Apriori), phân lớp (cây quyết định, Perceptron, MLP), phân cụm (K-Means). Mỗi dạng có **cài đặt thủ công** và **gọi hàm thư viện**.

**Bài 2 — Tiền xử lý:** chia Train/Test, chuẩn hóa StandardScaler và MinMaxScaler.

**Cách sử dụng:** chạy từng ô theo thứ tự trong mỗi dạng bài. Các dạng bài độc lập nhau. Dùng Python 3.10+ với `numpy pandas scikit-learn`; riêng Apriori bằng thư viện cần `mlxtend`. Cài bằng:

```bash
python -m pip install numpy pandas scikit-learn mlxtend notebook
```

**Quy tắc:** với thuật toán phân lớp, chia Train/Test trước khi chuẩn hóa, chỉ `fit` bộ chuẩn hóa trên tập Train. Với cây quyết định, chuẩn hóa thường không cần thiết. Apriori và K-Means không cần nhãn `y` trong bài cơ bản.

## BÀI 1 — Dạng 1: Luật kết hợp (Apriori)

- **Đầu vào:** danh sách giao dịch; mỗi giao dịch là một tập mặt hàng.
- **Kết quả:** các tập mục phổ biến và luật dạng `A → B`.
- `support(A ∪ B)`: tỷ lệ giao dịch chứa cả A và B.
- `confidence(A → B) = support(A ∪ B) / support(A)`.
- `lift(A → B) = confidence(A → B) / support(B)`.

### 1A. Cài đặt thủ công từng bước Apriori

In [ ]:
from itertools import combinations
import pandas as pd

# THAM SỐ ĐẦU VÀO — đổi danh sách này theo đề bài.
transactions = [
    {'Sữa', 'Bánh mì', 'Bơ'},
    {'Sữa', 'Bánh mì'},
    {'Sữa', 'Bơ'},
    {'Bánh mì', 'Bơ'},
    {'Sữa', 'Bánh mì', 'Bơ'},
    {'Sữa', 'Bánh mì'},
]
MIN_SUPPORT = 0.33     # float từ 0 đến 1: ngưỡng hỗ trợ tối thiểu (33%).
MIN_CONFIDENCE = 0.60  # float từ 0 đến 1: ngưỡng tin cậy tối thiểu (60%).
MAX_LEN = 3            # int >= 1: số mặt hàng lớn nhất trong một tập mục.

def manual_apriori(giao_dich, min_support=0.3, max_len=None):
    # giao_dich: list[set[str]], mỗi set là một giao dịch.
    # min_support: tỷ lệ xuất hiện tối thiểu của tập mục (0..1).
    # max_len: kích thước tập mục tối đa; None nghĩa là không giới hạn.
    transactions_set = [set(t) for t in giao_dich]
    n = len(transactions_set)
    if n == 0:
        raise ValueError('Danh sách giao dịch không được rỗng')
    if not (0 < min_support <= 1):
        raise ValueError('min_support phải nằm trong (0, 1]')
    # Bước 1: lập C1 — các ứng viên gồm đúng 1 mặt hàng.
    candidates = {frozenset([item]) for tran in transactions_set for item in tran}
    all_frequent = {}
    k = 1
    while candidates and (max_len is None or k <= max_len):
        # Bước 2: quét giao dịch, tính support ứng viên Ck.
        counts = {c: sum(c.issubset(tran) for tran in transactions_set) for c in candidates}
        # Bước 3: lọc Lk theo min_support.
        frequent_k = {c: count / n for c, count in counts.items() if count / n >= min_support}
        if not frequent_k:
            break
        all_frequent.update(frequent_k)
        # Bước 4: nối tập mục Lk để tạo ứng viên C(k+1) và tỉa theo tính chất Apriori.
        k += 1
        keys = list(frequent_k)
        candidates = set()
        for a, b in combinations(keys, 2):
            union = a | b
            if len(union) == k and all(frozenset(s) in frequent_k for s in combinations(union, k - 1)):
                candidates.add(union)
    return all_frequent

def generate_rules(frequent_itemsets, min_confidence=0.6):
    # frequent_itemsets: dict[frozenset, float support] từ manual_apriori.
    # min_confidence: ngưỡng confidence tối thiểu (0..1).
    rules = []
    for itemset, support_ab in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        for r in range(1, len(itemset)):
            for a_tuple in combinations(sorted(itemset), r):
                A = frozenset(a_tuple)
                B = itemset - A
                support_a = frequent_itemsets[A]
                support_b = frequent_itemsets[B]
                conf = support_ab / support_a
                if conf >= min_confidence:
                    rules.append({
                        'A': ', '.join(sorted(A)),
                        'B': ', '.join(sorted(B)),
                        'support': round(support_ab, 3),
                        'confidence': round(conf, 3),
                        'lift': round(conf / support_b, 3),
                    })
    return pd.DataFrame(rules).sort_values(['confidence', 'lift'], ascending=False).reset_index(drop=True) if rules else pd.DataFrame(columns=['A', 'B', 'support', 'confidence', 'lift'])

frequent = manual_apriori(transactions, min_support=MIN_SUPPORT, max_len=MAX_LEN)
print('TẬP MỤC PHỔ BIẾN:')
for itemset, support in sorted(frequent.items(), key=lambda x: (len(x[0]), sorted(x[0]))):
    print(sorted(itemset), 'support =', round(support, 3))
print('\nLUẬT KẾT HỢP:')
print(generate_rules(frequent, min_confidence=MIN_CONFIDENCE).to_string(index=False))


### 1B. Apriori bằng thư viện `mlxtend`

`TransactionEncoder` chuyển danh sách giao dịch thành bảng True/False. `apriori` tìm tập mục phổ biến. `association_rules` tạo các luật và tính độ đo. Cài `mlxtend` nếu ô dưới báo thiếu thư viện.

In [ ]:
import pandas as pd
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError:
    print('Chưa cài mlxtend. Chạy: python -m pip install mlxtend')
else:
    transactions = [
        ['Sữa', 'Bánh mì', 'Bơ'], ['Sữa', 'Bánh mì'], ['Sữa', 'Bơ'],
        ['Bánh mì', 'Bơ'], ['Sữa', 'Bánh mì', 'Bơ'], ['Sữa', 'Bánh mì'],
    ]
    te = TransactionEncoder()
    df = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
    frequent = apriori(
        df,                # DataFrame: từng cột là mặt hàng, từng hàng là giao dịch True/False.
        min_support=0.33,  # float (0..1]: support tối thiểu, ví dụ 0.33 = 33%.
        use_colnames=True, # bool: True hiện tên mặt hàng, False hiện mã cột.
        max_len=3,         # int hoặc None: chiều dài tối đa của tập mục.
    )
    rules = association_rules(
        frequent,            # DataFrame: kết quả do apriori trả về.
        metric='confidence', # str: 'confidence', 'lift' hoặc 'support'.
        min_threshold=0.60,  # float: ngưỡng tối thiểu của metric đã chọn.
        num_itemsets=len(df) # int: số giao dịch; bắt buộc trong một số bản mlxtend mới.
    )
    print('TẬP MỤC:')
    print(frequent.to_string(index=False))
    print('\nLUẬT KẾT HỢP:')
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))


## BÀI 1 — Dạng 2: Phân lớp bằng cây quyết định

- **Đầu vào:** tập đặc trưng `X` và nhãn `y`.
- **Nguyên lý:** chọn thuộc tính tạo mức giảm entropy lớn nhất (Information Gain); chia nhánh đệ quy.
- **Chương trình thủ công** phù hợp khi dữ liệu là thuộc tính **rời rạc**.

### 2A. Cây quyết định ID3 thủ công

In [ ]:
import math
from collections import Counter
from pprint import pprint
import pandas as pd

# DỮ LIỆU VÍ DỤ: mỗi dòng là (Trời, Gió, Chơi?). Thay theo bảng đề bài.
data = pd.DataFrame([
    ('Nắng', 'Nhẹ', 'Không'), ('Nắng', 'Mạnh', 'Không'),
    ('Âm u', 'Nhẹ', 'Có'), ('Âm u', 'Mạnh', 'Có'),
    ('Mưa', 'Nhẹ', 'Có'), ('Mưa', 'Mạnh', 'Không'),
    ('Nắng', 'Nhẹ', 'Không'), ('Mưa', 'Nhẹ', 'Có'),
], columns=['Trời', 'Gió', 'Chơi'])
TARGET = 'Chơi'  # str: tên cột nhãn mà cây cần dự đoán.

def entropy(labels):
    # labels: danh sách/Series các nhãn lớp.
    counts = Counter(labels)
    n = len(labels)
    return -sum((v/n) * math.log2(v/n) for v in counts.values()) if n else 0.0

def id3(df, features, target):
    # df: DataFrame huấn luyện; features: list[str] tên các cột đặc trưng.
    # target: str tên cột nhãn.
    labels = df[target]
    majority = labels.mode().iloc[0]
    if labels.nunique() == 1 or not features:
        return majority
    base_entropy = entropy(labels)
    gains = {}
    for f in features:
        conditional_entropy = sum(
            len(group) / len(df) * entropy(group[target])
            for _, group in df.groupby(f)
        )
        gains[f] = base_entropy - conditional_entropy
    best = max(gains, key=gains.get)
    node = {'feature': best, 'default': majority, 'branches': {}}
    for value, group in df.groupby(best):
        remain = [f for f in features if f != best]
        node['branches'][value] = id3(group, remain, target)
    return node

def id3_predict(tree, sample):
    # tree: cây do id3() tạo; sample: dict các tên cột -> giá trị của mẫu mới.
    while isinstance(tree, dict):
        value = sample.get(tree['feature'])
        tree = tree['branches'].get(value, tree['default'])
    return tree

tree = id3(data, features=['Trời', 'Gió'], target=TARGET)
print('CÂY ID3:')
pprint(tree, sort_dicts=False)
print('DỰ ĐOÁN:', id3_predict(tree, {'Trời': 'Mưa', 'Gió': 'Nhẹ'}))


### 2B. Cây quyết định bằng `DecisionTreeClassifier`

Sử dụng bộ dữ liệu Iris (150 mẫu, 4 đặc trưng, 3 lớp). Không cần chuẩn hóa vì cây dựa trên ngưỡng tách đặc trưng.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                # X: đặc trưng; y: nhãn tương ứng từng dòng.
    test_size=0.20,      # float: tỷ lệ Test = 20%; Train = 80%.
    random_state=42,     # int: cố định kết quả chia khi chạy lại.
    shuffle=True,        # bool: trộn dữ liệu trước khi chia.
    stratify=y,          # Series nhãn: giữ gần đúng tỷ lệ từng lớp ở Train/Test.
)
model = DecisionTreeClassifier(
    criterion='entropy', # str: 'entropy', 'gini' hoặc 'log_loss' — tiêu chí chia.
    max_depth=3,         # int hoặc None: độ sâu tối đa; None không giới hạn.
    min_samples_split=2, # int >=2: số mẫu tối thiểu để một nút được xem xét chia.
    min_samples_leaf=1,  # int >=1: số mẫu tối thiểu tại mỗi nút lá.
    random_state=42,     # int: giúp tái lập kết quả.
)
model.fit(X_train, y_train) # Huấn luyện chỉ trên Train.
y_pred = model.predict(X_test) # Dự đoán nhãn của Test.
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=load_iris().target_names))
print('CÂY QUYẾT ĐỊNH:')
print(export_text(model, feature_names=list(X.columns)))


## BÀI 1 — Dạng 3: Mạng nơ-ron 1 lớp (Perceptron)

**Quy ước:** perceptron đơn lớp có lớp đầu ra, **không có lớp ẩn**. Đây là mô hình tuyến tính, học bằng quy tắc cập nhật trọng số khi dự đoán sai. Nếu đề yêu cầu "mạng có *1 lớp ẩn*", hãy chuyển sang `MLPClassifier(hidden_layer_sizes=(16,))` ở dạng 4.

### 3A. Perceptron thủ công — học cổng AND

In [ ]:
import numpy as np

X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)  # shape (4,2): 2 đầu vào.
y = np.array([0,0,0,1], dtype=int)                     # shape (4,): nhãn AND.
LEARNING_RATE = 0.1  # float >0: tốc độ học (mỗi lần cập nhật trọng số).
EPOCHS = 20          # int >0: số lượt duyệt toàn bộ dữ liệu.

weights = np.zeros(X.shape[1]) # Hai trọng số, ban đầu bằng 0.
bias = 0.0                    # Hệ số chệch, ban đầu bằng 0.
for epoch in range(EPOCHS):
    errors = 0
    for xi, true_label in zip(X, y):
        score = np.dot(xi, weights) + bias
        pred = int(score >= 0) # Hàm kích hoạt bước: score >= 0 thì dự đoán 1.
        error = true_label - pred
        weights += LEARNING_RATE * error * xi # Quy tắc học Perceptron.
        bias += LEARNING_RATE * error
        errors += abs(error)
    if errors == 0:
        print('Hội tụ tại epoch:', epoch + 1)
        break
print('Trọng số:', weights, '| Bias:', round(bias, 3))
print('Dự đoán AND:', [int(np.dot(x, weights) + bias >= 0) for x in X])


### 3B. Perceptron thư viện — phân loại Iris

Perceptron nhạy với tỷ lệ các đặc trưng, vì vậy dùng `Pipeline(StandardScaler(), Perceptron(...))`: scaler chỉ học tham số từ Train, tránh rò rỉ dữ liệu.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model = make_pipeline(
    StandardScaler(),             # Chuẩn hóa z-score: (x - mean_train) / std_train.
    Perceptron(
        max_iter=1000,           # int: tối đa số epoch.
        eta0=0.1,                # float >0: learning rate.
        tol=1e-3,                # float hoặc None: ngưỡng dừng khi không cải thiện.
        random_state=42,         # int: cố định kết quả ngẫu nhiên.
        shuffle=True,            # bool: xáo trộn mẫu giữa các epoch.
    )
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy Perceptron:', round(accuracy_score(y_test, y_pred), 4))


## BÀI 1 — Dạng 4: Mạng nơ-ron nhiều lớp (MLP)

**Nguyên lý:** đi xuôi (forward) tính dự đoán, tính hàm mất mát, lan truyền ngược (backpropagation) và cập nhật trọng số bằng gradient descent.

### 4A. MLP thủ công (2 lớp có trọng số: 1 ẩn + 1 đầu ra) — học XOR

In [ ]:
import numpy as np

# Dữ liệu XOR: [0,0]->0; [0,1]->1; [1,0]->1; [1,1]->0.
X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float) # shape (4,1): đầu ra nhị phân.
HIDDEN_NEURONS = 8  # int >0: số nơ-ron trong lớp ẩn.
EPOCHS = 15000       # int >0: số lần cập nhật toàn bộ tập huấn luyện.
LEARNING_RATE = 1.5  # float >0: bước cập nhật gradient (tùy dữ liệu phải điều chỉnh).
SEED = 42            # int: cố định việc khởi tạo trọng số.

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

rng = np.random.default_rng(SEED)
W1 = rng.normal(0, 0.8, (X.shape[1], HIDDEN_NEURONS))
b1 = np.zeros((1, HIDDEN_NEURONS))
W2 = rng.normal(0, 0.8, (HIDDEN_NEURONS, 1))
b2 = np.zeros((1, 1))

for epoch in range(EPOCHS):
    # Bước 1: forward — truyền dữ liệu từ đầu vào -> ẩn -> đầu ra.
    a1 = sigmoid(X @ W1 + b1)
    a2 = sigmoid(a1 @ W2 + b2)
    # Bước 2: backprop — gradient của binary cross-entropy + sigmoid ở đầu ra.
    dz2 = (a2 - y) / len(X)
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)
    dz1 = (dz2 @ W2.T) * a1 * (1 - a1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)
    # Bước 3: cập nhật trọng số, bias.
    W2 -= LEARNING_RATE * dW2
    b2 -= LEARNING_RATE * db2
    W1 -= LEARNING_RATE * dW1
    b1 -= LEARNING_RATE * db1

prob = sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2)
print('Xác suất dự đoán:', np.round(prob.ravel(), 3))
print('Nhãn dự đoán XOR:', (prob.ravel() >= 0.5).astype(int))


### 4B. MLP thư viện — phân loại Iris

`hidden_layer_sizes=(16, 8)` nghĩa là **2 lớp ẩn**: lớp ẩn 1 có 16 nơ-ron, lớp ẩn 2 có 8 nơ-ron. Lớp đầu ra tự được tạo theo số lớp nhãn. Nếu đề muốn **1 lớp ẩn**, thay bằng `(16,)` (phải có dấu phẩy).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model = make_pipeline(
    StandardScaler(), # Chuẩn hóa đặc trưng theo thống kê của X_train.
    MLPClassifier(
        hidden_layer_sizes=(16, 8), # tuple: số nơ-ron MỖI lớp ẩn; (16,) = 1 lớp ẩn.
        activation='relu',         # str: 'relu', 'logistic', 'tanh' (hàm kích hoạt).
        solver='adam',             # str: 'adam', 'sgd', 'lbfgs' (thuật toán tối ưu).
        alpha=0.0001,              # float >=0: hệ số chính quy hóa L2.
        learning_rate_init=0.01,   # float >0: tốc độ học ban đầu cho adam/sgd.
        max_iter=2000,             # int: giới hạn số epoch cho adam/sgd.
        early_stopping=False,      # bool: True sử dụng tập validation để dừng sớm.
        random_state=42,          # int: giúp tái lập kết quả.
    )
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy MLP:', round(accuracy_score(y_test, y_pred), 4))
print('F1-macro MLP:', round(f1_score(y_test, y_pred, average='macro'), 4))


## BÀI 1 — Dạng 5: Phân cụm K-Means

**Đầu vào:** `X` (không cần nhãn). **Các bước:** chọn K tâm ban đầu → gán mỗi điểm vào tâm gần nhất → cập nhật tâm bằng trung bình → lặp đến khi hội tụ.

### 5A. K-Means thủ công

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

X, _ = make_blobs(
    n_samples=150,   # int: số điểm dữ liệu mô phỏng.
    centers=3,       # int: số cụm dùng để PHÁT SINH dữ liệu mô phỏng.
    cluster_std=0.8, # float: mức phân tán dữ liệu quanh tâm mô phỏng.
    random_state=42, # int: hạt giống ngẫu nhiên.
)
X = StandardScaler().fit_transform(X)  # Ở bài phân cụm cơ bản, fit trên toàn X.
K = 3              # int: số cụm muốn phân dữ liệu thành.
MAX_ITER = 100     # int: số vòng lặp tối đa.
TOL = 1e-4         # float: ngưỡng dừng nếu tâm dịch chuyển rất nhỏ.
SEED = 42          # int: cố định việc chọn tâm ban đầu.

def manual_kmeans(X, n_clusters=3, max_iter=100, tol=1e-4, random_state=42):
    # X: ndarray 2D (n_samples, n_features), chỉ chứa các giá trị số.
    # n_clusters: int từ 1 tới n_samples, số cụm K.
    # max_iter: int dương, số vòng lặp tối đa.
    # tol: float >=0, ngưỡng dịch chuyển tâm để dừng.
    # random_state: int hoặc None, cố định việc chọn tâm ban đầu.
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or not (1 <= n_clusters <= len(X)):
        raise ValueError('X phải là ma trận 2 chiều và 1 <= K <= số mẫu')
    rng = np.random.default_rng(random_state)
    centers = X[rng.choice(len(X), size=n_clusters, replace=False)].copy()
    for iteration in range(max_iter):
        # Bước 1: tính khoảng cách Euclid từ mỗi điểm tới từng tâm.
        distances = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        # Bước 2: gán vào cụm có tâm gần nhất.
        labels = np.argmin(distances, axis=1)
        # Bước 3: cập nhật tâm; nếu cụm trống, giữ nguyên tâm cũ.
        new_centers = np.array([
            X[labels == i].mean(axis=0) if np.any(labels == i) else centers[i]
            for i in range(n_clusters)
        ])
        # Bước 4: kiểm tra điều kiện dừng.
        if np.max(np.linalg.norm(new_centers - centers, axis=1)) < tol:
            centers = new_centers
            break
        centers = new_centers
    labels = np.argmin(np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2), axis=1)
    return labels, centers, iteration + 1

labels, centers, iters = manual_kmeans(X, K, MAX_ITER, TOL, SEED)
print('Số vòng lặp:', iters)
print('Số điểm ở mỗi cụm:', np.bincount(labels, minlength=K))
print('Tâm cụm:\n', centers)


### 5B. K-Means thư viện

`KMeans` trả về `labels_` là nhãn cụm (0, 1, 2, ...), `cluster_centers_` là tọa độ tâm cụm, `inertia_` là tổng bình phương khoảng cách tới tâm gần nhất. `silhouette_score` càng gần 1 càng tốt trong nhiều tình huống; chỉ tính được khi có ít nhất 2 cụm và ít cụm hơn số mẫu.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=150, centers=3, cluster_std=0.8, random_state=42)
X_scaled = StandardScaler().fit_transform(X)
model = KMeans(
    n_clusters=3,       # int: số cụm K muốn chia.
    init='k-means++',   # str: 'k-means++' hoặc 'random' — cách khởi tạo tâm.
    n_init=10,          # int: số lần thử khởi tạo tâm; chọn kết quả inertia nhỏ nhất.
    max_iter=300,       # int: số vòng lặp tối đa mỗi lần khởi tạo.
    tol=1e-4,           # float: ngưỡng hội tụ.
    random_state=42,    # int: giúp kết quả tái lập.
)
labels = model.fit_predict(X_scaled) # Huấn luyện và lấy nhãn cụm.
print('Nhãn 10 mẫu đầu:', labels[:10])
print('Tâm cụm:\n', model.cluster_centers_)
print('Inertia:', round(model.inertia_, 3))
print('Silhouette:', round(silhouette_score(X_scaled, labels), 3))


# BÀI 2 — Chuẩn hóa và chia Train/Test

## 6A. Chia dữ liệu Train/Test

Dùng cho bài toán **phân lớp có giám sát** (Decision Tree, Perceptron, MLP). `stratify=y` giữ gần đúng tỷ lệ các lớp. Nếu lớp chỉ có 1 mẫu, phải xử lý trước khi chia phân tầng.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                # X: ma trận đặc trưng (n_mẫu, n_thuộc_tính); y: nhãn lớp.
    test_size=0.2,       # float 0..1: tỷ lệ Test, ở đây 20%.
    # train_size=0.8,    # float 0..1: tỷ lệ Train; không cần ghi nếu đã có test_size.
    random_state=42,     # int hoặc None: cố định ngẫu nhiên để tái lập kết quả.
    shuffle=True,        # bool: True trộn các hàng trước khi chia.
    stratify=y,          # mảng nhãn y: chia phân tầng, giữ tỷ lệ từng lớp.
)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('y_train:', y_train.shape, '| y_test:', y_test.shape)


## 6B. Chuẩn hóa Z-score và Min-Max

- **StandardScaler:** `(x - mean_train) / std_train` → trung bình gần 0 và độ lệch chuẩn gần 1 trên Train.
- **MinMaxScaler:** đưa về khoảng `[0, 1]` tính trên Train (Test có thể vượt khoảng này nếu nằm ngoài min/max của Train).
- **Điểm quan trọng:** KHÔNG `fit_transform(X_test)` vì gây rò rỉ dữ liệu và dùng thang đo khác Train.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# CÁCH 1: Z-score — thường phù hợp với mạng nơ-ron, SVM, KNN.
standard = StandardScaler(
    with_mean=True, # bool: trừ trung bình của từng cột (dữ liệu dense).
    with_std=True,  # bool: chia cho độ lệch chuẩn của từng cột.
)
X_train_std = standard.fit_transform(X_train) # HỌC mean/std từ TRAIN rồi biến đổi TRAIN.
X_test_std = standard.transform(X_test)       # Chỉ biến đổi TEST bằng mean/std đã học.

# CÁCH 2: Min-Max — đưa Train về đoạn [0,1].
minmax = MinMaxScaler(
    feature_range=(0, 1), # tuple (min, max): khoảng mục tiêu, chẳng hạn (0,1) hay (-1,1).
    clip=False,            # bool: True cắt giá trị Test vượt phạm vi huấn luyện.
)
X_train_mm = minmax.fit_transform(X_train)
X_test_mm = minmax.transform(X_test)
print('Trước chuẩn hóa (mẫu Train đầu):', X_train[0])
print('Sau StandardScaler:', X_train_std[0].round(3))
print('Sau MinMaxScaler:', X_train_mm[0].round(3))
print('Kích thước Test sau chuẩn hóa:', X_test_std.shape)


## 6C. Kết hợp chuẩn hóa + phân lớp bằng `Pipeline` (khuyến nghị)

`Pipeline` tự thực hiện `fit_transform` trên Train và `transform` trên Test. Dùng tốt với Perceptron/MLP; Decision Tree không bắt buộc chuẩn hóa.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = Pipeline(steps=[
    ('scaler', StandardScaler()),  # Bước 1: chuẩn hóa; tự fit trên Train.
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(16,), # 1 lớp ẩn với 16 nơ-ron.
        max_iter=1500,             # số vòng lặp tối đa.
        random_state=42,          # cố định ngẫu nhiên.
    )),
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy mô hình Pipeline:', round(accuracy_score(y_test, y_pred), 4))


## BẢNG GHI NHỚ THAM SỐ

| Hàm / thuật toán | Tham số thường thay trong bài thi | Ý nghĩa |
|---|---|---|
| `apriori` | `min_support=0.3` | Tập mục xuất hiện ít nhất trong 30% giao dịch |
| `association_rules` | `metric='confidence'`, `min_threshold=0.7` | Chọn luật confidence ít nhất 70% |
| `DecisionTreeClassifier` | `criterion='entropy'`, `max_depth=3` | Chia theo entropy, giới hạn độ sâu 3 |
| `Perceptron` | `eta0=0.1`, `max_iter=1000` | Bước học và số epoch tối đa |
| `MLPClassifier` | `hidden_layer_sizes=(16,8)` | Hai lớp ẩn có 16 và 8 nơ-ron |
| `KMeans` | `n_clusters=3`, `n_init=10` | K=3, thử 10 lần khởi tạo tâm |
| `train_test_split` | `test_size=0.2`, `stratify=y` | 80% Train/20% Test, giữ tỷ lệ lớp |
| `StandardScaler` | `with_mean=True`, `with_std=True` | Trừ trung bình và chia độ lệch chuẩn |
| `MinMaxScaler` | `feature_range=(0,1)` | Đưa đặc trưng về khoảng mục tiêu |

**Hướng thay dữ liệu thực tế:** thay `load_iris(...)` bằng `df = pd.read_csv('ten_file.csv')`, `X = df.drop(columns=['ten_cot_nhan'])`, `y = df['ten_cot_nhan']` (với các bài phân lớp); nếu có cột chữ, cần mã hóa dữ liệu phù hợp trước khi đưa vào mô hình.